# SpatialData Write Smoke Test

This notebook reproduces the minimal `spatialdata`-only write test.

It does not use `sopa` or any real image data. It only:
1. creates a tiny synthetic image,
2. wraps it in a `SpatialData` object,
3. attempts `SpatialData.write(...)`,
4. uses `faulthandler` to show where the process is stuck if the write hangs.


In [1]:
from __future__ import annotations

import os
import shutil
import faulthandler
from pathlib import Path

os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba_cache")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")

import numpy as np
import spatialdata
from spatialdata import SpatialData
from spatialdata.models import Image2DModel

WRITE_PATH = Path("/tmp/spatialdata_write_smoke_test.sdata.zarr")
DUMP_AFTER_SECONDS = 20

print({"spatialdata": spatialdata.__version__, "write_path": str(WRITE_PATH)})


{'spatialdata': '0.7.2', 'write_path': '/tmp/spatialdata_write_smoke_test.sdata.zarr'}


/home/ratnayn/miniconda3/envs/spatialdata/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Build a Tiny Synthetic SpatialData Object


In [2]:
if WRITE_PATH.exists():
    shutil.rmtree(WRITE_PATH)

tiny_array = np.random.default_rng(0).random((3, 64, 64), dtype=np.float32)
tiny_image = Image2DModel.parse(
    tiny_array,
    dims=("c", "y", "x"),
    c_coords=["channel_a", "channel_b", "channel_c"],
)
tiny_sdata = SpatialData(images={"img": tiny_image})

print({
    "array_shape": tiny_array.shape,
    "array_dtype": str(tiny_array.dtype),
    "parsed_chunks": tiny_image.data.chunks,
})
tiny_sdata


{'array_shape': (3, 64, 64), 'array_dtype': 'float32', 'parsed_chunks': ((3,), (64,), (64,))}


SpatialData object
└── Images
      └── 'img': DataArray[cyx] (3, 64, 64)
with coordinate systems:
    ▸ 'global', with elements:
        img (Images)

In [3]:
tiny_sdata

SpatialData object
└── Images
      └── 'img': DataArray[cyx] (3, 64, 64)
with coordinate systems:
    ▸ 'global', with elements:
        img (Images)

## Attempt the Write

If the write hangs, `faulthandler` will print a traceback after `DUMP_AFTER_SECONDS` seconds.


In [4]:
print(f"About to write to {WRITE_PATH}")
faulthandler.dump_traceback_later(DUMP_AFTER_SECONDS, repeat=False)
try:
    tiny_sdata.write(WRITE_PATH, overwrite=True)
finally:
    faulthandler.cancel_dump_traceback_later()

print("Write returned successfully")


About to write to /tmp/spatialdata_write_smoke_test.sdata.zarr
Write returned successfully


In [5]:
import spatialdata, ome_zarr, zarr, dask
print("spatialdata", spatialdata.__version__)
print("ome_zarr", ome_zarr.__version__)
print("zarr", zarr.__version__)
print("dask", dask.__version__)

spatialdata 0.7.2
ome_zarr 0.13.0
zarr 3.1.6
dask 2026.1.1


## Inspect the Output Store


In [6]:
print({
    "exists": WRITE_PATH.exists(),
    "children": sorted(path.name for path in WRITE_PATH.iterdir()) if WRITE_PATH.exists() else [],
})


{'exists': True, 'children': ['images', 'zarr.json']}


In [7]:
import spatialdata
import zarr
import ome_zarr

print("spatialdata", spatialdata.__version__)
print("ome_zarr", ome_zarr.__version__)
print("zarr", zarr.__version__)

spatialdata 0.7.2
ome_zarr 0.13.0
zarr 3.1.6


In [9]:
from spatialdata import read_zarr

reloaded = read_zarr(WRITE_PATH)
print(reloaded)
print("images", list(reloaded.images.keys()))
print("labels", list(reloaded.labels.keys()))
print("tables", list(reloaded.tables.keys()))


SpatialData object, with associated Zarr store: /tmp/spatialdata_write_smoke_test.sdata.zarr
└── Images
      └── 'img': DataArray[cyx] (3, 64, 64)
with coordinate systems:
    ▸ 'global', with elements:
        img (Images)
images ['img']
labels []
tables []
